In [1]:
import os
import pickle
import pandas as pd
from metient.util import plotting_util as plutil
from metient.util import data_extraction_util as dutil
from metient.util.globals import *

In [2]:
# Change this to your local path where the reproducibility data is stored
DATA_DIR = "/data1/morrisq/divyak/data/metient_reproducibility/nsclc"
METIENT_OUTPUT_DIR = os.path.join(DATA_DIR, 'metient_outputs')

sample_info_df= pd.read_csv(os.path.join(DATA_DIR,"sample_overview_original.txt"), sep="\t")
sample_info_df

,patient_id,tumour_id,region,sampleType,sampleTypeDetail
0,CRUK0010,CRUK0010,CRUK0010_SU_T1.R1,primary,primary
1,CRUK0010,CRUK0010,CRUK0010_SU_T1.R2,primary,primary
2,CRUK0010,CRUK0010,CRUK0010_SU_FLN1,metastasis,LN
3,CRUK0010,CRUK0010,CRUK0010_BR_LN1,metastasis,metachronousMet
4,CRUK0010,CRUK0010,CRUK0010_BR_LN2,metastasis,metachronousMet
...,...,...,...,...,...
689,CRUK0872,CRUK0872,CRUK0872_SU_T1.R1,primary,primary
690,CRUK0872,CRUK0872,CRUK0872_SU_T1.R2,primary,primary
691,CRUK0872,CRUK0872,CRUK0872_SU_T1.R3,primary,primary
692,CRUK0872,CRUK0872,CRUK0872_SU_T1.R4,primary,primary


### How many patients have LN metastases?


In [3]:
len(sample_info_df[sample_info_df['sampleTypeDetail']=='LN']['patient_id'].unique())

96

In [4]:
import gzip
import torch

def get_patients(pickle_files_dir):
    patients = set()
    for file in os.listdir(pickle_files_dir):
        if ".pkl.gz" in file:
            name = file.split(".")[0].replace("_calibrate", "")
            patients.add(name)
    return patients

def patient_in_dict(dct, patient):
    for patient_primary in dct:
        if patient == patient_primary.split("_")[0]:
            return True, patient_primary, dct[patient_primary]
    return False, None, None

def get_seeding_sites(G, sites, patient, primary_site):
    pid = patient.split("_")[0]
    seeding_indices = [int(x) for x in torch.where(torch.sum(G,dim=1))[0]]
    seeding = ['_'.join(sites[x].split("_")[1:]) for x in seeding_indices]
    patient_sample_info = sample_info_df[sample_info_df['patient_id']==pid]
    seeding_sites = set()
    for _,row in patient_sample_info.iterrows():
        for s in seeding:
            if s in row['region']:
                seeding_sites.add(row['sampleTypeDetail'])
    prim_seeds_one_site = (torch.sum(G[sites.index(primary_site)] != 0) == 1).item()
    return list(seeding_sites), prim_seeds_one_site

def get_info(pickle_files_dir):
    patients = list(get_patients(pickle_files_dir))
    seeding_patterns = dict()
    for patient in patients:

        with gzip.open(os.path.join(pickle_files_dir, f"{patient}.pkl.gz"),'rb') as f:
            pckl = pickle.load(f)
        Vs = pckl[OUT_LABElING_KEY]
        As = pckl[OUT_PARENTS_KEY]
        idx_to_labels = pckl[OUT_IDX_LABEL_KEY]
        loss_dicts = pckl[OUT_LOSS_DICT_KEY]
        losses = [l.item() for l in pckl[OUT_LOSSES_KEY]]
        sites = pckl[OUT_SITES_KEY]
        primary_site = pckl[OUT_PRIMARY_KEY]
        site_clonalities, gen_clonalities, patterns, phyleticities, tracerx_phyleticites = [],[],[],[],[]
        Gs = []
        gen_dist_losses = []
        seeding_sites = []
        prim_seeds_one_sites = []
        for x, (V, A, loss_dict, idx_to_label) in enumerate(zip(Vs, As, loss_dicts, idx_to_labels)):

            V = torch.tensor(V)
            A = torch.tensor(dutil.adjacency_matrix_from_parents(A))
            
            phyleticity = plutil.phyleticity(V, A, idx_to_label)
            tracerx_phyleticity = plutil.tracerx_phyleticity(V, A, idx_to_label)
            site_clonality = plutil.site_clonality(V, A)
            gen_clonality = plutil.genetic_clonality(V, A, idx_to_label)
            pattern = plutil.seeding_pattern(V, A)
            G = plutil.migration_graph(V,A)
            x,y = get_seeding_sites(G,sites,patient,primary_site)
            seeding_sites.append(x)
            prim_seeds_one_sites.append(y)
            phyleticities.append(phyleticity)
            tracerx_phyleticites.append(tracerx_phyleticity)
            site_clonalities.append(site_clonality)
            gen_clonalities.append(gen_clonality)
            patterns.append(pattern)
            Gs.append(G)
            gen_dist_losses.append(float(loss_dict[GEN_DIST_KEY]))
        # For patients with multiple primaries
        patient_name = patient.split("_")[0]
        if patient_name in seeding_patterns:
            prev_best_loss = seeding_patterns[patient_name][8][0]
            current_best_loss = losses[0]
            # choose the run with the lower loss
            if current_best_loss < prev_best_loss:
                seeding_patterns[patient_name] = site_clonalities, gen_clonalities, patterns, phyleticities, tracerx_phyleticites, Gs, gen_dist_losses, seeding_sites, losses, prim_seeds_one_sites
        else:
            seeding_patterns[patient_name] = site_clonalities, gen_clonalities, patterns, phyleticities, tracerx_phyleticites, Gs, gen_dist_losses, seeding_sites, losses, prim_seeds_one_sites

    print(f"{len(seeding_patterns)} patients")
    return seeding_patterns

### Load metient outputs

In [5]:
conipher_mp_gd_seeding_patterns = get_info(METIENT_OUTPUT_DIR)
inferred_patterns = conipher_mp_gd_seeding_patterns
print(len(inferred_patterns))

/data1/morrisq/divyak/projects/metient/metient/util/data_extraction_util.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  parents = torch.tensor(parents)[mask]
/tmp/ipykernel_1338982/1047760067.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A = torch.tensor(dutil.adjacency_matrix_from_parents(A))


126 patients
126


### How many patients have different migration graphs in their Pareto front with met-to-met seeding

In [6]:
mult_nonpss_mig_graphs = set()

def add_tensor_to_set(tensor_set, new_tensor):
    for tensor in tensor_set:
        if torch.equal(tensor, new_tensor):
            return
    tensor_set.append(new_tensor)

pids_w_mult_ss= set()

for patient in inferred_patterns:
    seeding_sites = inferred_patterns[patient][7][0]
    prim_seeds_one_site = inferred_patterns[patient][9][0]
    if len(seeding_sites) > 1:
        print(patient,seeding_sites, prim_seeds_one_site)
            
        pids_w_mult_ss.add(patient)

print("Patients with met-to-met seeding:", len(pids_w_mult_ss), pids_w_mult_ss)


for patient in inferred_patterns:
    Gs = inferred_patterns[patient][5]
    gen_dist_losses = inferred_patterns[patient][6]
    if len(Gs) == 1: continue
    not_pss_Gs = []
    for G in Gs:
        mult_seeding_sites = len(torch.nonzero(G.sum(axis=1))) > 1
        if mult_seeding_sites:
            add_tensor_to_set(not_pss_Gs, G)
    
    if len(not_pss_Gs) > 1:
        mult_nonpss_mig_graphs.add(patient)

print("Patients with multiple unique mig graphs with met-to-met seeding", len(mult_nonpss_mig_graphs), mult_nonpss_mig_graphs)

CRUK0256 ['primary', 'LN'] True
CRUK0495 ['primary', 'synchronousMet'] False
CRUK0748 ['primary', 'LN'] False
CRUK0736 ['primary', 'LN'] True
CRUK0290 ['primary', 'LN'] True
CRUK0468 ['primary', 'LN'] True
CRUK0484 ['primary', 'metachronousMet'] False
CRUK0590 ['primary', 'metachronousMet'] False
CRUK0810 ['primary', 'LN'] True
CRUK0620 ['primary', 'LN'] True
CRUK0090 ['primary', 'metachronousMet'] True
CRUK0487 ['primary', 'LN'] False
CRUK0465 ['primary', 'LN'] False
CRUK0311 ['primary', 'LN'] False
CRUK0242 ['primary', 'metachronousMet'] True
CRUK0559 ['primary', 'LN'] True
CRUK0245 ['primary', 'LN'] False
CRUK0698 ['primary', 'LN'] True
CRUK0029 ['primary', 'LN'] False
CRUK0013 ['primary', 'LN'] True
Patients with met-to-met seeding: 20 {'CRUK0487', 'CRUK0620', 'CRUK0013', 'CRUK0748', 'CRUK0311', 'CRUK0698', 'CRUK0256', 'CRUK0090', 'CRUK0242', 'CRUK0484', 'CRUK0468', 'CRUK0810', 'CRUK0465', 'CRUK0736', 'CRUK0590', 'CRUK0029', 'CRUK0495', 'CRUK0559', 'CRUK0245', 'CRUK0290'}
Patients 

### How concordant are our seeding patterns with TRACERx's?

In [7]:
tracerx_seeding = pd.read_csv(os.path.join(DATA_DIR, "seedingTable.txt"), sep="\t")
tracerx_timing = pd.read_csv(os.path.join(DATA_DIR, "timingTable.txt"), sep="\t")

tracerx_seeding.columns = ["patient_id", "tracerx_clonality","tracerx_phyletic", "tracerx_multitree_adjustment"] 
print(len(tracerx_seeding))

def get_metient_patterns(row, pattern_idx):
    patient_id = row['patient_id'].split("_")[0]
    return ",".join(inferred_patterns[patient_id][pattern_idx])
    
tracerx_seeding['metient_genetic_clonalities'] = tracerx_seeding.apply(lambda row: get_metient_patterns(row, 1), axis=1)
tracerx_seeding['metient_tracerx_phyletics'] = tracerx_seeding.apply(lambda row: get_metient_patterns(row, 4), axis=1)
tracerx_seeding['metient_patterns'] = tracerx_seeding.apply(lambda row: get_metient_patterns(row, 2), axis=1)
tracerx_seeding['tracerx_timing'] = tracerx_seeding.apply(lambda row: tracerx_timing[tracerx_timing['tumour_id']==row['patient_id']]['timing'].item(), axis=1)
tracerx_seeding['metient_site_clonalities'] = tracerx_seeding.apply(lambda row: get_metient_patterns(row, 0), axis=1)

tracerx_seeding['metient_site_clonality_first'] = tracerx_seeding.apply(lambda row: row['metient_site_clonalities'].split(",")[0], axis=1)
tracerx_seeding['metient_genetic_clonality_first'] = tracerx_seeding.apply(lambda row: row['metient_genetic_clonalities'].split(",")[0], axis=1)
tracerx_seeding['metient_tracerx_phyletic_first'] = tracerx_seeding.apply(lambda row: row['metient_tracerx_phyletics'].split(",")[0], axis=1)

tracerx_seeding


126


,patient_id,tracerx_clonality,tracerx_phyletic,tracerx_multitree_adjustment,metient_genetic_clonalities,metient_tracerx_phyletics,metient_patterns,tracerx_timing,metient_site_clonalities,metient_site_clonality_first,metient_genetic_clonality_first,metient_tracerx_phyletic_first
0,CRUK0010,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,late,monoclonal,monoclonal,polyclonal,polyphyletic
1,CRUK0013,monoclonal,monophyletic,monophyletic,"polyclonal,polyclonal","monophyletic,monophyletic","single-source,primary single-source",late,"monoclonal,polyclonal",monoclonal,polyclonal,monophyletic
2,CRUK0284,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,late,monoclonal,monoclonal,polyclonal,polyphyletic
3,CRUK0361,monoclonal,monophyletic,monophyletic,monoclonal,monophyletic,primary single-source,late,monoclonal,monoclonal,monoclonal,monophyletic
4,CRUK0497,monoclonal,monophyletic,monophyletic,monoclonal,monophyletic,primary single-source,late,monoclonal,monoclonal,monoclonal,monophyletic
...,...,...,...,...,...,...,...,...,...,...,...,...
121,CRUK0495,polyclonal,monophyletic,monophyletic,"polyclonal,polyclonal","polyphyletic,polyphyletic","multi-source,primary single-source",late,"polyclonal,polyclonal",polyclonal,polyclonal,polyphyletic
122,CRUK0476,monoclonal,monophyletic,monophyletic,monoclonal,monophyletic,primary single-source,late,monoclonal,monoclonal,monoclonal,monophyletic
123,CRUK0528,polyclonal,polyphyletic,polyphyletic,polyclonal,polyphyletic,primary single-source,late,polyclonal,polyclonal,polyclonal,polyphyletic
124,CRUK0666,polyclonal,polyphyletic,polyphyletic,polyclonal,polyphyletic,primary single-source,late,polyclonal,polyclonal,polyclonal,polyphyletic


In [8]:
set(tracerx_seeding['patient_id']) - set(inferred_patterns.keys())

{'CRUK0084_Tumour2',
 'CRUK0301_Tumour1',
 'CRUK0372_Tumour1',
 'CRUK0620_Tumour1',
 'CRUK0721_Tumour1'}

### How concordant are our characterizations of monophyletic/polyphyletic with TRACERx (using their definition of phyleticity)

In [9]:
    
num_tracerx_mono_met_mono = len(tracerx_seeding[(tracerx_seeding['tracerx_multitree_adjustment']=="monophyletic")&(tracerx_seeding['metient_tracerx_phyletic_first']=="monophyletic")])
print("Phyletic consensus")
print("tracerx mono met mono", (num_tracerx_mono_met_mono))

tracerx_mono_met_poly = tracerx_seeding[(tracerx_seeding['tracerx_multitree_adjustment']=="monophyletic")&(tracerx_seeding['metient_tracerx_phyletic_first']=="polyphyletic")]
num_tracerx_mono_met_poly = len(tracerx_mono_met_poly)
print("tracerx mono met poly", (num_tracerx_mono_met_poly), list(tracerx_mono_met_poly['patient_id']))

num_tracerx_poly_met_poly = len(tracerx_seeding[(tracerx_seeding['tracerx_multitree_adjustment']=="polyphyletic")&(tracerx_seeding['metient_tracerx_phyletic_first']=="polyphyletic")])
print("tracerx poly met poly", (num_tracerx_poly_met_poly))

tracerx_poly_met_mono = tracerx_seeding[(tracerx_seeding['tracerx_multitree_adjustment']=="polyphyletic")&(tracerx_seeding['metient_tracerx_phyletic_first']=="monophyletic")]
num_tracerx_poly_met_mono = len(tracerx_poly_met_mono)
print("tracerx poly met mono", (num_tracerx_poly_met_mono), list(tracerx_poly_met_mono['patient_id']))

total = num_tracerx_mono_met_mono+num_tracerx_mono_met_poly+num_tracerx_poly_met_poly+num_tracerx_poly_met_mono
print("total", total)
data = [["Monophyletic", 100*(num_tracerx_mono_met_mono/total)], 
        ["Polyphyletic",100*(num_tracerx_poly_met_poly/total)], 
        ["Metient: Polyphyletic, \nTRACERx: Monophyletic",100*(num_tracerx_mono_met_poly/total)],
        ["Metient: Monophyletic, \nTRACERx: Polyphyletic",100*(num_tracerx_poly_met_mono/total)],]



Phyletic consensus
tracerx mono met mono 81
tracerx mono met poly 26 ['CRUK0010', 'CRUK0284', 'CRUK0472', 'CRUK0702', 'CRUK0590', 'CRUK0620_Tumour1', 'CRUK0609', 'CRUK0714', 'CRUK0290', 'CRUK0745', 'CRUK0087', 'CRUK0299', 'CRUK0530', 'CRUK0325', 'CRUK0090', 'CRUK0029', 'CRUK0035', 'CRUK0372_Tumour1', 'CRUK0487', 'CRUK0467', 'CRUK0514', 'CRUK0557', 'CRUK0559', 'CRUK0762', 'CRUK0810', 'CRUK0495']
tracerx poly met poly 16
tracerx poly met mono 0 []
total 123


In [10]:
print(tracerx_seeding['metient_tracerx_phyletic_first'].value_counts())
print(tracerx_seeding['metient_tracerx_phyletic_first'].value_counts(normalize=True) * 100)

metient_tracerx_phyletic_first
monophyletic    83
polyphyletic    43
Name: count, dtype: int64
metient_tracerx_phyletic_first
monophyletic   65.873
polyphyletic   34.127
Name: proportion, dtype: float64


In [11]:
tracerx_seeding[tracerx_seeding['tracerx_multitree_adjustment']=='mixed']

,patient_id,tracerx_clonality,tracerx_phyletic,tracerx_multitree_adjustment,metient_genetic_clonalities,metient_tracerx_phyletics,metient_patterns,tracerx_timing,metient_site_clonalities,metient_site_clonality_first,metient_genetic_clonality_first,metient_tracerx_phyletic_first
24,CRUK0352,polyclonal,monophyletic,mixed,polyclonal,monophyletic,primary single-source,late,polyclonal,polyclonal,polyclonal,monophyletic
29,CRUK0478,polyclonal,monophyletic,mixed,polyclonal,monophyletic,primary single-source,late,polyclonal,polyclonal,polyclonal,monophyletic
61,CRUK0245,polyclonal,monophyletic,mixed,"polyclonal,polyclonal,polyclonal,polyclonal,po...","polyphyletic,polyphyletic,polyphyletic,polyphy...","multi-source,multi-source,multi-source,multi-s...",late,"polyclonal,polyclonal,polyclonal,polyclonal,po...",polyclonal,polyclonal,polyphyletic


In [12]:
clonality_type = 'metient_genetic_clonality_first'

num_tracerx_mono_met_mono = len(tracerx_seeding[(tracerx_seeding['tracerx_clonality']=="monoclonal")&(tracerx_seeding[clonality_type]=="monoclonal")])
print("tracerx mono met mono", (num_tracerx_mono_met_mono))
tracerx_mono_met_poly = tracerx_seeding[(tracerx_seeding['tracerx_clonality']=="monoclonal")&(tracerx_seeding[clonality_type]=="polyclonal")]
num_tracerx_mono_met_poly = len(tracerx_mono_met_poly)
print("tracerx mono met poly", (num_tracerx_mono_met_poly), set(tracerx_mono_met_poly['patient_id']))
num_tracerx_poly_met_poly = len(tracerx_seeding[(tracerx_seeding['tracerx_clonality']=="polyclonal")&(tracerx_seeding[clonality_type]=="polyclonal")])
print("tracerx poly met poly", (num_tracerx_poly_met_poly))
tracerx_poly_met_mono = tracerx_seeding[(tracerx_seeding['tracerx_clonality']=="polyclonal")&(tracerx_seeding[clonality_type]=="monoclonal")]
num_tracerx_poly_met_mono = len(tracerx_poly_met_mono)
print("tracerx poly met mono", (num_tracerx_poly_met_mono), set(tracerx_poly_met_mono['patient_id']))


data = [["Monoclonal", num_tracerx_mono_met_mono], 
        ["Polyclonal",num_tracerx_poly_met_poly], 
        ["Metient: Polyclonal, \nTRACERx: Monoclonal",num_tracerx_mono_met_poly],
        ["Metient: Monoclonal, \nTRACERx: Polyclonal",num_tracerx_poly_met_mono],]
df = pd.DataFrame( data,columns=["Method", "Number of cases"])
print(df)


tracerx mono met mono 66
tracerx mono met poly 20 {'CRUK0702', 'CRUK0572', 'CRUK0013', 'CRUK0557', 'CRUK0698', 'CRUK0452', 'CRUK0584', 'CRUK0530', 'CRUK0284', 'CRUK0256', 'CRUK0090', 'CRUK0242', 'CRUK0468', 'CRUK0810', 'CRUK0745', 'CRUK0087', 'CRUK0559', 'CRUK0714', 'CRUK0299', 'CRUK0010'}
tracerx poly met poly 40
tracerx poly met mono 0 set()
                                       Method  Number of cases
0                                  Monoclonal               66
1                                  Polyclonal               40
2  Metient: Polyclonal, \nTRACERx: Monoclonal               20
3  Metient: Monoclonal, \nTRACERx: Polyclonal                0


In [13]:
tracerx_seeding[(tracerx_seeding['tracerx_clonality']=="monoclonal")&(tracerx_seeding[clonality_type]=="polyclonal")]

,patient_id,tracerx_clonality,tracerx_phyletic,tracerx_multitree_adjustment,metient_genetic_clonalities,metient_tracerx_phyletics,metient_patterns,tracerx_timing,metient_site_clonalities,metient_site_clonality_first,metient_genetic_clonality_first,metient_tracerx_phyletic_first
0,CRUK0010,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,late,monoclonal,monoclonal,polyclonal,polyphyletic
1,CRUK0013,monoclonal,monophyletic,monophyletic,"polyclonal,polyclonal","monophyletic,monophyletic","single-source,primary single-source",late,"monoclonal,polyclonal",monoclonal,polyclonal,monophyletic
2,CRUK0284,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,late,monoclonal,monoclonal,polyclonal,polyphyletic
5,CRUK0452,monoclonal,monophyletic,monophyletic,polyclonal,monophyletic,primary single-source,late,monoclonal,monoclonal,polyclonal,monophyletic
10,CRUK0702,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,late,monoclonal,monoclonal,polyclonal,polyphyletic
21,CRUK0256,monoclonal,monophyletic,monophyletic,"polyclonal,polyclonal","monophyletic,monophyletic","single-source,primary single-source",late,"monoclonal,polyclonal",monoclonal,polyclonal,monophyletic
41,CRUK0698,monoclonal,monophyletic,monophyletic,"polyclonal,polyclonal,polyclonal","monophyletic,monophyletic,monophyletic","single-source,single-source,primary single-source",late,"polyclonal,polyclonal,polyclonal",polyclonal,polyclonal,monophyletic
45,CRUK0714,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,early,monoclonal,monoclonal,polyclonal,polyphyletic
51,CRUK0468,monoclonal,monophyletic,monophyletic,"polyclonal,polyclonal","monophyletic,monophyletic","single-source,primary single-source",late,"monoclonal,polyclonal",monoclonal,polyclonal,monophyletic
55,CRUK0745,monoclonal,monophyletic,monophyletic,polyclonal,polyphyletic,primary single-source,late,polyclonal,polyclonal,polyclonal,polyphyletic


In [14]:
tracerx_poly_met_mono = tracerx_seeding[(tracerx_seeding['tracerx_multitree_adjustment']=="polyphyletic")&(tracerx_seeding['metient_tracerx_phyletics'][0]=="monophyletic")]
tracerx_poly_met_mono_pids = list(tracerx_poly_met_mono['patient_id'].unique())
print(len(tracerx_poly_met_mono_pids))


from collections import Counter
num_seeding_sites = []
for pid in tracerx_poly_met_mono_pids:
    if pid in inferred_patterns:
        best_G = inferred_patterns[pid][5][0]
        non_zero_rows = torch.unique(best_G.nonzero()[:, 0])
        num_non_zero_rows = len(non_zero_rows)
        num_seeding_sites.append(num_non_zero_rows)

ctr = Counter(num_seeding_sites)
ctr
    

0


Counter()